# Setup

In [1]:
from pprint import pprint
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Seed
seed = 42

# sklearn config
from sklearn import set_config
set_config(transform_output='pandas', display='text')

# Data

In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

X, y = fetch_california_housing(
    return_X_y=True,
    as_frame=True,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=True,
    random_state=seed,
)

# 1. Grid Search

```python
class GridSearchCV(
    estimator: BaseEstimator,
    param_grid: Mapping | list[Mapping],
    *,
    scoring: str | Callable | list | tuple | dict | None = None,
    n_jobs: int | None = None,
    refit: bool | str | Callable = True,
    cv: int | BaseCrossValidator | Iterable | None = None,
    verbose: int = 0,
    pre_dispatch: int | str = "2*n_jobs",
    error_score: float | str = np.nan,
    return_train_score: bool = False
)
```

In [3]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor

estimator = DecisionTreeRegressor()

param_grid = {
    'ccp_alpha': np.arange(0.0, 0.5, 0.25),     # [0.0, 0.25].
    'splitter': ['random', 'best'],
}

search = GridSearchCV(
    estimator=estimator,
    param_grid=param_grid,
    scoring=None,
    cv=2,
    refit=True,     # after search, automatically refit using the entire dataset.
    verbose=2,
)

search.fit(X_train, y_train)

model = search.best_estimator_
search.best_params_

Fitting 2 folds for each of 4 candidates, totalling 8 fits
[CV] END .....................ccp_alpha=0.0, splitter=random; total time=   0.0s
[CV] END .....................ccp_alpha=0.0, splitter=random; total time=   0.0s
[CV] END .......................ccp_alpha=0.0, splitter=best; total time=   0.0s
[CV] END .......................ccp_alpha=0.0, splitter=best; total time=   0.0s
[CV] END ....................ccp_alpha=0.25, splitter=random; total time=   0.1s
[CV] END ....................ccp_alpha=0.25, splitter=random; total time=   0.1s
[CV] END ......................ccp_alpha=0.25, splitter=best; total time=   0.1s
[CV] END ......................ccp_alpha=0.25, splitter=best; total time=   0.1s


{'ccp_alpha': np.float64(0.0), 'splitter': 'best'}

# 2. Random Search

```python
class RandomizedSearchCV(
    estimator: BaseEstimator,
    param_distributions: Mapping | list[Mapping],
    *,
    n_iter: int = 10,
    scoring: str | Callable | list | tuple | dict | None = None,
    n_jobs: int | None = None,
    refit: bool | str | Callable = True,
    cv: int | BaseCrossValidator | Iterable | None = None,
    verbose: int = 0,
    pre_dispatch: int | str = "2*n_jobs",
    random_state: int | RandomState | None = None,
    error_score: float | str = np.nan,
    return_train_score: bool = False
)
```

In [4]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.tree import DecisionTreeRegressor

estimator = DecisionTreeRegressor()

param_distributions = {
    'ccp_alpha': np.arange(0.0, 0.5, 0.25),     # [0.0, 0.25].
    'splitter': ['random', 'best'],
}

search = RandomizedSearchCV(
    estimator=estimator,
    param_distributions=param_distributions,
    n_iter=2,   # randomly choose 2 from 2x2 = 4 combinations.
    scoring=None,
    cv=2,
    refit=True,
    verbose=2,
    random_state=seed,
)

search.fit(X_train, y_train)

model = search.best_estimator_

search.best_params_

Fitting 2 folds for each of 2 candidates, totalling 4 fits
[CV] END .......................ccp_alpha=0.0, splitter=best; total time=   0.0s
[CV] END .......................ccp_alpha=0.0, splitter=best; total time=   0.0s
[CV] END ......................ccp_alpha=0.25, splitter=best; total time=   0.1s
[CV] END ......................ccp_alpha=0.25, splitter=best; total time=   0.1s


{'splitter': 'best', 'ccp_alpha': np.float64(0.0)}

# 3. Optuna

## 1) Basic

In [5]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor

# 1. Objective
def objective(trial):
    params = {
        'splitter': trial.suggest_categorical(
            'splitter', ['random', 'best'],
        ),
        'ccp_alpha': trial.suggest_float(   # [0.0, 0.25, 0.5].
            name='ccp_alpha',
            low=0.0,
            high=0.5,
            step=0.25,
            log=False,
        ),
    }
    model = DecisionTreeRegressor(**params)
    score = cross_val_score(
        model,
        X_train,
        y_train,
        cv=2,
    ).mean()

    return score

# 2. Study
sampler = optuna.samplers.TPESampler(seed=seed)
storage = 'sqlite:///C:/Users/yana/Desktop/Library/AI/ai-summary/optuna.db'      # no local path but sqlite.
pruner = optuna.pruners.MedianPruner()

study = optuna.create_study(
    study_name='test',          # study name.
    storage=storage,            # None for memory-only.
    load_if_exists=True,        # if True, load if exists.
    direction='maximize',       # maximize/minimize the objective.
    sampler=sampler,            # sample for each trial.
    pruner=pruner,              # early-stopping.
)

# 3. Optimize
study.optimize(
    func=objective,             # objective function.
    n_trials=2,                 # maximum number of combinations to try.
    timeout=None,               # maximum total optimization time in seconds.
    n_jobs=1,                   # number of trials to run in parallel.
    catch=(),                   # exception types that do not stop the study.
    callbacks=None,             # callbacks called after each trial.
    gc_after_trial=True,        # if True, run gc after every trial.
    show_progress_bar=True,     # show progress bar.
)

print(f"Best params: {study.best_params}")
print(f"Best value: {study.best_value}")

# 4. Refit
model = DecisionTreeRegressor(**study.best_params)
model.fit(X_train, y_train)

# 5. Delete (if necessary)
# optuna.delete_study(study_name='test', storage=storage)

c:\Users\yana\Desktop\Library\AI\ai-summary\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-10-06 17:59:40,564] Using an existing study with `study_name='test'` instead of creating a new one.
Best trial: 0. Best value: 0.530703:  50%|█████     | 1/2 [00:00<00:00,  1.44it/s]

[I 2026-10-06 17:59:41,186] Trial 2 finished with value: -0.00014841945275478885 and parameters: {'splitter': 'best', 'ccp_alpha': 0.5}. Best is trial 0 with value: 0.5307027153136479.
[I 2026-10-06 17:59:41,341] Trial 3 finished with value: 0.5085274289730186 and parameters: {'splitter': 'random', 'ccp_alpha': 0.0}. Best is trial 0 with value: 0.5307027153136479.


Best trial: 0. Best value: 0.530703: 100%|██████████| 2/2 [00:00<00:00,  2.35it/s]

Best params: {'splitter': 'random', 'ccp_alpha': 0.0}
Best value: 0.5307027153136479


DecisionTreeRegressor(splitter='random')

## 2) Search Space

In [6]:
def objective(trial):
    params = {
        # Categorical
        'splitter': trial.suggest_categorical(
            'splitter', ['random', 'best'],
        ),

        # Boolean
        'bootstrap': trial.suggest_categorical(
            'bootstrap', [True, False],
        ),

        # Float
        'ccp_alpha': trial.suggest_float(   # [0.0, 0.25, 0.5].
            name='ccp_alpha',
            low=0.0,
            high=0.5,
            step=0.25,
            log=False,
        ),

        # Int
        'max_depth': trial.suggest_int(
            name='max_trial',
            low=2,
            high=3,
            step=1,
        )
    }

    # Conditional by other parameters.
    if params['splitter'] == 'random':
        params['ccp_alpha'] = trial.suggest_float('ccp_alpha', 0.0, 0.1)

    ...

## 3) Samplers

| Sampler          | Main idea                                                                                         | Typical use                                                |
| ---------------- | ------------------------------------------------------------------------------------------------- | ---------------------------------------------------------- |
| `TPESampler`     | Uses previous trials to model promising vs. poor regions and samples more from promising regions. | Default and most important general-purpose choice      |
| `RandomSampler`  | Samples hyperparameters independently at random.                                                  | Baseline, debugging, simple search                         |
| `GridSampler`    | Exhaustively evaluates predefined combinations.                                                   | Small discrete search spaces                               |
| `CmaEsSampler`   | Evolutionary optimization specialized for continuous numerical parameters.                        | Mostly continuous search spaces                            |
| `QMCSampler`     | Uses quasi-random low-discrepancy sequences instead of ordinary random sampling.                  | More even coverage of continuous spaces                    |
| `GPSampler`      | Gaussian-process-based Bayesian optimization.                                                     | Expensive objectives with relatively few dimensions/trials |
| `NSGAIISampler`  | Evolutionary algorithm for multiple objectives.                                                   | Multi-objective optimization                               |
| `NSGAIIISampler` | Extension of NSGA-II designed especially for many objectives.                                     | Multi-objective problems with several objectives           |


## 4) Pruning

| Pruner                    | Idea                                                                              |
| ------------------------- | --------------------------------------------------------------------------------- |
| `MedianPruner`            | Prune trials performing poorly relative to previous trials. **Learn this first.** |
| `SuccessiveHalvingPruner` | Progressively allocate more resources only to promising trials.                   |
| `HyperbandPruner`         | Runs multiple successive-halving schedules with different resource allocations.   |


In [7]:
import optuna

from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split


X_subtrain, X_valid, y_subtrain, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=seed,
)


def objective(trial):
    params = {
        'alpha': trial.suggest_float(
            'alpha',
            1e-6,
            1e-1,
            log=True,
        ),
    }

    model = SGDRegressor(**params)

    for step in range(100):

        model.partial_fit(X_subtrain, y_subtrain)

        y_pred = model.predict(X_valid)
        score = mean_squared_error(y_valid, y_pred)

        trial.report(score, step)

        if trial.should_prune():
            raise optuna.TrialPruned()

    return score

sampler = optuna.samplers.TPESampler(seed=seed)
pruner = optuna.pruners.MedianPruner()

study = optuna.create_study(
    study_name='test_prune',    # study name.
    storage=None,               # None for memory-only.
    load_if_exists=True,        # if True, load if exists.
    direction='minimize',       # maximize/minimize the objective.
    sampler=sampler,            # sample for each trial.
    pruner=pruner,              # early-stopping.
)

study.optimize(
    objective,
    n_trials=5,
)

[I 2026-10-06 17:59:41,473] A new study created in memory with name: test_prune
[I 2026-10-06 17:59:41,796] Trial 0 finished with value: 2.9774707887101145e+29 and parameters: {'alpha': 7.459343285726558e-05}. Best is trial 0 with value: 2.9774707887101145e+29.
[I 2026-10-06 17:59:42,126] Trial 1 finished with value: 1.64473158705072e+29 and parameters: {'alpha': 0.05669849511478854}. Best is trial 1 with value: 1.64473158705072e+29.
[I 2026-10-06 17:59:42,445] Trial 2 finished with value: 7.404144465386632e+29 and parameters: {'alpha': 0.004570563099801453}. Best is trial 1 with value: 1.64473158705072e+29.
[I 2026-10-06 17:59:42,770] Trial 3 finished with value: 2.7768469216691103e+29 and parameters: {'alpha': 0.0009846738873614568}. Best is trial 1 with value: 1.64473158705072e+29.
[I 2026-10-06 17:59:43,089] Trial 4 finished with value: 2.239244636133475e+28 and parameters: {'alpha': 6.026889128682509e-06}. Best is trial 4 with value: 2.239244636133475e+28.


## 5) Inspection

### Values

In [8]:
study.best_params
study.best_value
study.best_trial
study.trials

df = study.trials_dataframe()

display(df)

,number,value,datetime_start,datetime_complete,duration,params_alpha,state
0,0,2.977471e+29,2026-10-06 17:59:41.474174,2026-10-06 17:59:41.796599,0 days 00:00:00.322425,0.000075,COMPLETE
1,1,1.644732e+29,2026-10-06 17:59:41.797185,2026-10-06 17:59:42.126332,0 days 00:00:00.329147,0.056698,COMPLETE
2,2,7.404144e+29,2026-10-06 17:59:42.126976,2026-10-06 17:59:42.445511,0 days 00:00:00.318535,0.004571,COMPLETE
3,3,2.776847e+29,2026-10-06 17:59:42.446147,2026-10-06 17:59:42.770842,0 days 00:00:00.324695,0.000985,COMPLETE
4,4,2.239245e+28,2026-10-06 17:59:42.771462,2026-10-06 17:59:43.089763,0 days 00:00:00.318301,0.000006,COMPLETE


### Visualization

| API                           | What you inspect                                                 |
| ----------------------------- | ---------------------------------------------------------------- |
| `plot_optimization_history()` | Optimization progress and where good trials occurred             |
| `plot_param_importances()`    | Which hyperparameters most affected the objective                |
| `plot_slice()`                | Relationship between each hyperparameter value and the objective |
| `plot_contour()`              | Interaction between pairs of hyperparameters                     |
| `plot_intermediate_values()`  | Intermediate scores and pruning behavior                         |

> Note) For interactive inspection, run below:

```markdown
optuna-dashboard sqlite:///optuna.db --storage-class RDBStorage
```

In [9]:
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_slice,
    plot_contour,
    plot_intermediate_values,
)

plot_optimization_history(study).show()

plot_param_importances(study).show()

plot_slice(study).show()

plot_contour(study).show()

plot_intermediate_values(study).show()

## 6) Reproduction

### `user_attr`

In [10]:
study.set_user_attr('seed', 42)
study.set_user_attr('dataset', 'california_housing')
study.set_user_attr('metric', 'r2')

study.user_attrs

{'seed': 42, 'dataset': 'california_housing', 'metric': 'r2'}

### Load

In [ ]:
study = optuna.load_study(
    study_name='test',
    storage=storage,
)